# MOM6 closure: global heat, salt and mass conservation

Does the ocean (MOM6) conserve heat, salt and mass globally? If it does, the change in global ocean heat content (OHC), salt content (OSC) and mass over any period must equal the surface fluxes into the ocean integrated over that period:

| Budget | Change in content | Fluxes in (globally integrated) |
|---|---|---|
| Heat | $\Delta \mathrm{OHC} = \Delta\left(c_p\,\overline{\Theta}\,M\right)$ | `hfds` |
| Salt | $\Delta \mathrm{OSC} = \Delta\left(10^{-3}\,\overline{S}\,M\right)$ | `sfdsi` + `salt_flux_added` |
| Mass | $\Delta M$ | `wfo` |

where $M$ is the mass of liquid ocean (`masso`), $\overline{\Theta}$ the global-mean conservative temperature (`bigthetaoga`) and $\overline{S}$ the global-mean salinity (`soga`).

**Sea ice.** These are ocean-only budgets, so sea ice enters only through what it exchanges with the ocean, and the surface flux diagnostics already include those exchanges:
- `hfds` is the net heat flux at the ocean surface, including the ice-ocean heat flux, frazil formation and sea-ice melt heat (see its `long_name` below).
- `wfo` includes the water from sea-ice melt and formation.
- `sfdsi` is the salt exchanged with sea ice as it melts and forms; `salt_flux_added` is surface salinity restoring.

The heat and salt held in the sea ice itself belong to the ocean+sea-ice budget, which checks fluxes between model components; that is a separate check.

**No 3D fields needed.** MOM6 writes daily snapshots of `masso`, `bigthetaoga` and `soga`, and in this Boussinesq configuration $c_p\,\overline{\Theta}\,M = \sum c_p\,\rho_0\,\Theta\,h\,A$ exactly, which is the model's own heat content. So OHC and OSC at every month boundary come from scalars, without loading `thetao`, `so` or `thkcello`.

Related: [`ocean_stats.ipynb`](ocean_stats.ipynb) plots MOM6's own per-timestep conservation errors from `ocean.stats.nc`. This notebook does the same check independently, from the saved diagnostics.

Issue: [access-om3-paper-1#62](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/62)

In [ ]:
# These first two cells must be in all notebooks!
# It allows us to run all the notebooks at once, this cell has a tag "parameters" which allows us to pass in
# arguments externally using papermill (see mkfigs.sh for details)

# Set esm_file to the datastore for the main experiment of interest
# e.g. https://access-om3-configs.access-hive.org.au/latest/Experiments/
esm_file = "/g/data/ol01/outputs/access-om3-25km/MC_25km_jra_iaf+wombatlite-test5.1-8f50b769/datastore.json"

# papermill settings. *No need to modify these if running interactively.*
papermill = False                      # `cwd` and `nbname` will be populated by papermill.
cwd = None                             # current working directory
nbname = None                          # notebook name

In [ ]:
if not papermill:
    import nci_ipynb, os  # requires conda/analysis3-26.03 or later
    cwd = nci_ipynb.dir()
    nbname = nci_ipynb.name()
    os.chdir(cwd)
import mkfigs_bootstrap  # noqa: adds external/access-model-mkfigs/src to sys.path (stop-gap)
from mkfigs import MkmdWriter
mkmd = MkmdWriter(esm_file, nbname, str(cwd), pm=papermill)

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import intake
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from distributed import Client

In [ ]:
client = Client(threads_per_worker=1)
print(client.dashboard_link)

### Open the intake-esm datastore

In [ ]:
COLUMNS_WITH_ITERABLES = [
        "variable",
        "variable_long_name",
        "variable_standard_name",
        "variable_cell_methods",
        "variable_units"
]

datastore = intake.open_esm_datastore(
    esm_file,
    columns_with_iterables=COLUMNS_WITH_ITERABLES
)

### The variables used in the budgets

In [ ]:
def available_variables(datastore):
    '''Return a pandas dataframe summarising the variables in a datastore'''
    variable_columns = [col for col in datastore.df.columns if "variable" in col]
    return (
        datastore.df[variable_columns]
        .explode(variable_columns)
        .drop_duplicates()
        .set_index("variable")
        .sort_index()
    )

In [ ]:
# Monthly-mean surface fluxes into the ocean, and the daily global scalars
flux_variables = ["hfds", "hfrainds", "hfrunoffds", "wfo", "sfdsi", "salt_flux_added"]
scalar_variables = ["masso", "bigthetaoga", "soga"]

budget_variables = available_variables(datastore.search(variable=flux_variables + scalar_variables))
budget_variables = budget_variables.loc[flux_variables + scalar_variables]
budget_variables[["variable_long_name", "variable_units"]]

### Model constants

$c_p$ and $\rho_0$ are the values the model ran with (`C_P` and `RHO_0` in `MOM_parameter_doc.all`). Check them against the run's parameter file if you change `esm_file`.

In [ ]:
C_P = 3992.0   # J kg-1 K-1, MOM6 C_P for this configuration
RHO_0 = 1035.0  # kg m-3, MOM6 RHO_0 (Boussinesq reference density; only used to quote mass as sea level)

### Load the monthly surface fluxes and integrate them over the globe

Each flux is multiplied by the cell area (`areacello`) and summed over the globe, giving W (heat) or kg s-1 (water, salt) for each month. This is done lazily with dask, one month per chunk, so only the global totals are held in memory.

In [ ]:
# Cell area from the static file.
# path is a workaround - see https://forum.access-hive.org.au/t/issues-in-loading-ht-in-latest-conds-envs/6278
path = datastore.search(variable="areacello", filename="access-om3.mom6.static.nc").df.loc[0, "path"]
areacello = datastore.search(variable="areacello", path=path).to_dask()["areacello"].fillna(0.0).compute()

ocean_area = float(areacello.sum())  # m2
print(f"Ocean area: {ocean_area:.4e} m2")

In [ ]:
# Global integral of each monthly-mean flux: W for heat fluxes, kg s-1 for water and salt fluxes.
# Chunks of one month and the full horizontal grid suit a horizontal sum done month by month.
global_flux = {}
for variable in flux_variables:
    flux = datastore.search(variable=variable, frequency="1mon").to_dask(
        xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
            compat="override",
            data_vars="minimal",
            coords="minimal",
        ),
        xarray_open_kwargs = dict(
            chunks={"time": 1, "yh": -1, "xh": -1},
            decode_timedelta=True
        )
    )[variable]
    global_flux[variable] = (flux.fillna(0.0) * areacello).sum(["yh", "xh"])

global_flux = xr.Dataset(global_flux).compute()
global_flux

### Turn the monthly-mean fluxes into totals for each month

Each monthly mean is multiplied by the number of seconds in that month. The run uses a Gregorian calendar, so February has 29 days in leap years; `days_in_month` handles this.

In [ ]:
seconds_in_month = global_flux.time.dt.days_in_month * 24 * 3600

heat_in = global_flux["hfds"] * seconds_in_month                                   # J per month
salt_in = (global_flux["sfdsi"] + global_flux["salt_flux_added"]) * seconds_in_month  # kg of salt per month
mass_in = global_flux["wfo"] * seconds_in_month                                    # kg of water per month

### Load the daily global scalars and compute content at the start of each month

`masso`, `bigthetaoga` and `soga` are daily **snapshots**, so the value stamped `YYYY-MM-01 00:00` is the state at the very start of that month, exactly the boundary of the monthly flux averaging period.

In [ ]:
scalars = datastore.search(variable=scalar_variables, frequency="1day").to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat="override",
        data_vars="minimal",
        coords="minimal",
    ),
)[scalar_variables].squeeze(drop=True).compute()

ocean_heat_content = C_P * scalars["bigthetaoga"] * scalars["masso"]   # J (heat relative to 0 degC)
ocean_salt_content = 1e-3 * scalars["soga"] * scalars["masso"]         # kg of salt
ocean_mass = scalars["masso"]                                          # kg

In [ ]:
# Pick the snapshots that fall on the first of each month.
# The first snapshot is at the end of day 1 (e.g. 1958-01-02), so the first month
# of the run has no starting snapshot and is dropped from the budget below.
month_start = pd.DatetimeIndex(global_flux.time.dt.strftime("%Y-%m-01").values)
next_month_start = month_start + pd.offsets.MonthBegin(1)

def change_over_each_month(content):
    '''Content at the start of the next month minus content at the start of this month.
    NaN where either snapshot is missing.'''
    content = content.assign_coords(time=content.indexes["time"].normalize())
    start = content.reindex(time=month_start).values
    end = content.reindex(time=next_month_start).values
    return xr.DataArray(end - start, coords={"time": global_flux.time}, dims="time")

heat_change = change_over_each_month(ocean_heat_content)
salt_change = change_over_each_month(ocean_salt_content)
mass_change = change_over_each_month(ocean_mass)

# Keep only the months with both snapshots
complete = np.isfinite(heat_change) & np.isfinite(salt_change) & np.isfinite(mass_change)
print(f"{int(complete.sum())} of {complete.size} months have snapshots at both ends")

### Budgets: content change against integrated surface fluxes

The residual (change in content minus integrated flux) is what the fluxes do not explain. A closed budget has a residual that is tiny compared with either term, with no drift.

In [ ]:
budget = xr.Dataset({
    "heat_change": heat_change, "heat_in": heat_in,
    "salt_change": salt_change, "salt_in": salt_in,
    "mass_change": mass_change, "mass_in": mass_in,
}).where(complete, drop=True)

for name in ["heat", "salt", "mass"]:
    budget[f"{name}_residual"] = budget[f"{name}_change"] - budget[f"{name}_in"]

budget

In [ ]:
# Cumulative content change and cumulative integrated flux since the first complete month
fig, axes = plt.subplots(3, 2, figsize=(15, 11), sharex=True)

panels = [
    ("heat", 1e21, "Heat (ZJ)"),
    ("salt", 1e15, "Salt (Pg)"),
    ("mass", 1e15, "Mass (Pg)"),
]

for row, (name, scale, label) in enumerate(panels):
    ax = axes[row, 0]
    (budget[f"{name}_change"].cumsum("time") / scale).plot(ax=ax, label="change in ocean content", lw=2)
    (budget[f"{name}_in"].cumsum("time") / scale).plot(ax=ax, label="integrated surface flux", ls="--")
    ax.set_ylabel(f"Cumulative {label}")
    ax.set_title(f"{name.capitalize()}: cumulative change vs fluxes")
    ax.set_xlabel("")
    ax.legend()

    ax = axes[row, 1]
    (budget[f"{name}_residual"].cumsum("time") / scale).plot(ax=ax, color="k")
    ax.axhline(0, color="grey", lw=0.5)
    ax.set_ylabel(f"Cumulative residual {label}")
    ax.set_title(f"{name.capitalize()}: content change minus fluxes")
    ax.set_xlabel("")

fig.tight_layout()

In [ ]:
# mkmd.savefig(fig, title, caption, dpi=100)
# Save figure and append to markdown summary.
mkmd.savefig(
    fig,
    "MOM6 closure: global heat, salt and mass",
    "Left: cumulative change in global ocean heat content (c_p * bigthetaoga * masso), salt content "
    "(soga * masso / 1000) and mass (masso), against the cumulative globally integrated surface fluxes "
    "(hfds; sfdsi + salt_flux_added; wfo). Right: cumulative residual (content change minus fluxes).",
    dpi=150,
)

### How well does each budget close?

For each budget: the typical monthly change in content, the typical monthly residual, their ratio, and the residual accumulated over the whole run expressed as an equivalent flux (W m-2 for heat, sea level for mass).

In [ ]:
n_seconds = float(seconds_in_month.where(complete, drop=True).sum())
units = {"heat": "J", "salt": "kg", "mass": "kg"}

rows = []
for name in ["heat", "salt", "mass"]:
    typical_change = float(np.abs(budget[f"{name}_change"]).mean())
    typical_residual = float(np.abs(budget[f"{name}_residual"]).mean())
    total_residual = float(budget[f"{name}_residual"].sum())
    rows.append({
        "budget": name,
        f"mean |monthly change|": f"{typical_change:.3e} {units[name]}",
        f"mean |monthly residual|": f"{typical_residual:.3e} {units[name]}",
        "residual / change": f"{typical_residual / typical_change:.1e}",
        "total residual over run": f"{total_residual:.3e} {units[name]}",
    })

# Express the accumulated residuals as equivalent global fluxes
heat_residual_wm2 = float(budget["heat_residual"].sum()) / n_seconds / ocean_area
mass_residual_sea_level_mm = float(budget["mass_residual"].sum()) / RHO_0 / ocean_area * 1e3
rows[0]["equivalent"] = f"{heat_residual_wm2:.2e} W m-2"
rows[1]["equivalent"] = ""
rows[2]["equivalent"] = f"{mass_residual_sea_level_mm:.2e} mm sea level"

closure_table = pd.DataFrame(rows).set_index("budget")
closure_table

In [ ]:
# mkmd.table(title, table)
# Append table to markdown summary.
mkmd.table("MOM6 closure: monthly budget residuals", closure_table.to_markdown().split("\n"))

### Where the heat goes in: the surface heat flux and its parts

`hfrainds` and `hfrunoffds` are the heat carried in by rain and runoff; they are already part of `hfds` (not extra terms), shown here for context.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4))
(global_flux["hfds"] / ocean_area).plot(ax=ax, label="hfds (net)")
(global_flux["hfrainds"] / ocean_area).plot(ax=ax, label="hfrainds (part of hfds)")
(global_flux["hfrunoffds"] / ocean_area).plot(ax=ax, label="hfrunoffds (part of hfds)")
ax.axhline(0, color="grey", lw=0.5)
ax.set_ylabel("Global-mean heat flux into ocean (W m-2)")
ax.set_xlabel("")
_ = ax.legend()

In [ ]:
client.close()